# HackAlem AI — рабочий ноутбук команды

Стартовая площадка для прототипа. Сначала выберите **один кейс**, прочитайте его обязательные требования и критерии, затем настройте нужные секции ниже. Этот шаблон не предполагает, что задача обязательно табличная: есть отдельные заготовки для табличных данных и LLM API.

**Правило команды:** сохраните основной пользовательский сценарий простым и проверяемым. Ключи API держите в локальном `.env`, никогда не вставляйте их в ноутбук или GitHub. Перед сдачей обновите README только тем, что действительно реализовано.

## 0. Окружение

Зависимости установлены из `requirements.txt` в локальную `.venv`. Если открыли ноутбук в другом окружении, установите их в терминале проекта командой `python -m pip install -r requirements.txt`, затем выберите kernel проекта.

In [ ]:
from pathlib import Path
import os, sys, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv

warnings.filterwarnings('ignore')
load_dotenv()
ROOT = Path.cwd()
DATA_DIR = ROOT / 'data'
OUTPUT_DIR = ROOT / 'outputs'
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)
print(f'Python {sys.version.split()[0]} | pandas {pd.__version__} | numpy {np.__version__}')
print('Project:', ROOT.resolve())
print('Data files:', [p.name for p in DATA_DIR.iterdir() if p.is_file()])

## 1. Карточка кейса и план на спринт

Заполните эту карточку после выбора кейса. Скопируйте обязательные требования из задания дословно, чтобы команда могла проверять готовность по ним.

In [ ]:
CASE_NAME = 'TODO: название кейса'
USER = 'TODO: кто будет пользоваться решением'
PROBLEM = 'TODO: какую задачу пользователя решаем'
INPUT_EXAMPLE = 'TODO: пример входа'
EXPECTED_OUTPUT = 'TODO: ожидаемый результат'
REQUIRED_ITEMS = [
    'TODO: обязательное требование 1',
    'TODO: обязательное требование 2',
]

for label, value in [('Кейс', CASE_NAME), ('Пользователь', USER), ('Проблема', PROBLEM), ('Вход', INPUT_EXAMPLE), ('Результат', EXPECTED_OUTPUT)]:
    print(f'{label}: {value}')
print('Проверки перед сдачей:')
for item in REQUIRED_ITEMS:
    print('[ ]', item)

## 2. Загрузка и быстрая проверка данных

Поместите доступные файлы в `data/`. CSV, JSON/JSONL и Parquet читаются автоматически. Для документов, изображений или аудио сначала выберите способ их обработки из требований кейса; этот блок только покажет список файлов. Не отправляйте закрытые данные во внешние сервисы без разрешения организаторов.

In [ ]:
def load_table(path: Path) -> pd.DataFrame:
    suffix = path.suffix.lower()
    if suffix == '.csv':
        return pd.read_csv(path)
    if suffix in {'.json', '.jsonl'}:
        return pd.read_json(path, lines=(suffix == '.jsonl'))
    if suffix in {'.parquet', '.pq'}:
        return pd.read_parquet(path)
    raise ValueError(f'Формат пока не поддержан: {suffix}')

table_paths = [p for p in DATA_DIR.iterdir() if p.is_file() and p.suffix.lower() in {'.csv', '.json', '.jsonl', '.parquet', '.pq'}]
if table_paths:
    DATA_PATH = table_paths[0]
    df = load_table(DATA_PATH)
    print(f'{DATA_PATH.name}: {df.shape[0]:,} строк × {df.shape[1]} столбцов')
    display(df.head())
    profile = pd.DataFrame({'dtype': df.dtypes.astype(str), 'missing': df.isna().sum(), 'unique': df.nunique(dropna=True)})
    display(profile.sort_values('missing', ascending=False))
else:
    print('Табличных файлов пока нет. Содержимое data/:', [p.name for p in DATA_DIR.iterdir() if p.is_file()])

In [ ]:
if 'df' in globals():
    numeric_cols = df.select_dtypes(include='number').columns.tolist()
    categorical_cols = df.select_dtypes(exclude='number').columns.tolist()
    print('Числовые столбцы:', numeric_cols)
    print('Текстовые/категориальные столбцы:', categorical_cols)
    if numeric_cols:
        display(df[numeric_cols].describe().T)
        df[numeric_cols].hist(figsize=(min(14, 4 * len(numeric_cols)), 3 * ((len(numeric_cols) + 3) // 4)), bins=25)
        plt.tight_layout(); plt.show()

## 3. Baseline для табличного ML (по необходимости)

Используйте только если кейс — классификация или числовой прогноз по таблице. Впишите название целевого столбца и тип задачи. Сначала получите простую метрику, затем сравните улучшения на одном и том же разбиении. Для другого типа кейса пропустите ячейку.

In [ ]:
TARGET = 'TODO: целевой столбец'
TASK = 'classification'  # или 'regression'

if 'df' not in globals():
    print('Загрузите табличные данные в секции 2.')
elif TARGET.startswith('TODO') or TARGET not in df.columns:
    print('Укажите TARGET из задания; baseline пока пропущен.')
elif TASK not in {'classification', 'regression'}:
    raise ValueError("TASK должен быть 'classification' или 'regression'")
else:
    from sklearn.model_selection import train_test_split
    from sklearn.compose import ColumnTransformer
    from sklearn.pipeline import Pipeline
    from sklearn.impute import SimpleImputer
    from sklearn.preprocessing import OneHotEncoder, StandardScaler
    from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
    from sklearn.metrics import accuracy_score, f1_score, mean_absolute_error

    X, y = df.drop(columns=[TARGET]), df[TARGET]
    num_cols = X.select_dtypes(include='number').columns.tolist()
    cat_cols = X.select_dtypes(exclude='number').columns.tolist()
    prep = ColumnTransformer([
        ('num', Pipeline([('fill', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), num_cols),
        ('cat', Pipeline([('fill', SimpleImputer(strategy='most_frequent')), ('encode', OneHotEncoder(handle_unknown='ignore'))]), cat_cols),
    ])
    estimator = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1) if TASK == 'classification' else RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
    model = Pipeline([('prep', prep), ('model', estimator)])
    stratify = y if TASK == 'classification' and y.value_counts().min() >= 2 else None
    X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42, stratify=stratify)
    model.fit(X_train, y_train)
    pred = model.predict(X_valid)
    if TASK == 'classification':
        print('Accuracy:', round(accuracy_score(y_valid, pred), 4))
        print('F1 macro:', round(f1_score(y_valid, pred, average='macro', zero_division=0), 4))
    else:
        print('MAE:', round(mean_absolute_error(y_valid, pred), 4))

## 4. Подключение языковой модели (опционально)

Для OpenAI задайте `OPENAI_API_KEY` в `.env`. Для NVIDIA задайте `NVIDIA_API_KEY` и точное имя модели из инструкции организаторов. Вызов API может расходовать квоту; запускайте его только когда ключи активированы и команда готова. Не вставляйте реальные ключи в ячейки.

In [ ]:
from openai import OpenAI

def ask_model(prompt: str, provider: str, model_name: str) -> str:
    if provider == 'openai':
        key = os.getenv('OPENAI_API_KEY')
        client = OpenAI(api_key=key) if key else None
    elif provider == 'nvidia':
        key = os.getenv('NVIDIA_API_KEY')
        client = OpenAI(api_key=key, base_url='https://integrate.api.nvidia.com/v1') if key else None
    else:
        raise ValueError("provider: 'openai' или 'nvidia'")
    if client is None:
        raise RuntimeError(f'Добавьте ключ для {provider} в локальный .env')
    result = client.chat.completions.create(model=model_name, messages=[{'role': 'user', 'content': prompt}], max_tokens=500)
    return result.choices[0].message.content or ''

# Пример: укажите провайдера и model_name из инструкции по активации.
# answer = ask_model('Кратко объясни идею проекта: ...', provider='openai', model_name='...')
# print(answer)

## 5. Минимальная приёмочная проверка

Для демонстрации жюри зафиксируйте один вход, ожидаемый результат и фактический результат. Ниже — шаблон, независимый от типа проекта. Добавьте проверки обязательных требований кейса.

In [ ]:
def check_demo(run_solution, demo_input, expected_check):
    actual = run_solution(demo_input)
    passed = bool(expected_check(actual))
    print('PASS' if passed else 'FAIL')
    print('Результат:', actual)
    return passed

# TODO: подключите здесь основную функцию вашего прототипа и повторяемый пример.
# check_demo(run_solution, demo_input, expected_check)

## Перед сдачей

- выбран один кейс; обязательные требования сверены с заданием и критериями;
- основной сценарий повторяется от входа до результата;
- ноутбук перезапущен сверху вниз;
- README объясняет задачу, реализованные функции, архитектуру, запуск, проверку, данные, сервисы и ограничения;
- в GitHub нет `.env`, ключей, приватных данных или лишних файлов;
- последний код загружен в командный репозиторий через Codex/VS Code или другой выбранный способ.